In [9]:
import math as m
import requests
import matplotlib.pyplot as plt
import numpy as np
import json

In [10]:
def haversine_distance(p1_latlon, p2_latlon, earth_radius=6371):

    p1_lat_rad = m.radians(float(p1_latlon.split(",")[0]))
    p1_lon_rad = m.radians(float(p1_latlon.split(",")[1]))
    p2_lat_rad = m.radians(float(p2_latlon.split(",")[0]))
    p2_lon_rad = m.radians(float(p2_latlon.split(",")[1]))

    d_lon = p2_lon_rad - p1_lon_rad
    d_lat = p2_lat_rad - p1_lat_rad

    a = m.sin(d_lat / 2)**2 + m.cos(p1_lat_rad) * m.cos(p2_lat_rad) * m.sin(d_lon / 2)**2
    c = 2 * m.asin(m.sqrt(a))

    total_d = c * earth_radius * 1000

    return total_d

def latlon_to_xy(p1_latlon, p2_latlon):

    p1_lat = p1_latlon.split(",")[0]
    p1_lon = p1_latlon.split(",")[1]
    p2_lat = p2_latlon.split(",")[0]
    p2_lon = p2_latlon.split(",")[1]

    p_y_latlon = p2_lat+","+p1_lon
    p_x_latlon = p1_lat+","+p2_lon

    p_2_y = haversine_distance(p1_latlon, p_y_latlon)
    p_2_x = haversine_distance(p1_latlon, p_x_latlon)
    p_2_xy = (p_2_x, p_2_y)

    return p_2_xy

In [11]:
def get_cells_from_opencellid(p_min_latlon, p_max_latlon, opencellid_token, radio_technology):

    cells_positions_xy = []

    if radio_technology == "None":
        request_str = "http://opencellid.org/cell/getInArea?key="+opencellid_token+"&BBOX="+p_min_latlon+","+p_max_latlon+"&mcc=214&mnc=07&format=json"

    else:
        request_str = "http://opencellid.org/cell/getInArea?key="+opencellid_token+"&BBOX="+p_min_latlon+","+p_max_latlon+"&radio="+radio_technology+"&mcc=214&format=json"

    response = requests.get(request_str)
    response_json = response.json()
    print(response_json)

    if "cells" in response_json:
        cells = response_json["cells"]  # list

        for i in range(0, len(cells)):  # Iterate cells
            current_cell = cells[i]
            current_cell_latlon_position = str(current_cell["lat"])+","+str(current_cell["lon"])
            current_xy_point = latlon_to_xy(p_min_latlon, current_cell_latlon_position)
            cells_positions_xy.append(current_xy_point)

        return cells_positions_xy, response_json

    else:
        return [], []

In [12]:
p1_grid_city_lonlat = ["39.51,-6.43","39.51,-6.41",
                       "39.51,-6.39","39.51,-6.37","39.51,-6.35","39.51,-6.33","39.49,-6.43","39.49,-6.41","39.49,-6.39","39.49,-6.37",
                       "39.49,-6.35","39.49,-6.33","39.47,-6.43","39.47,-6.41","39.47,-6.39","39.47,-6.37","39.47,-6.35","39.47,-6.33",
                       "39.45,-6.43","39.45,-6.41","39.45,-6.39","39.45,-6.37","39.45,-6.35","39.45,-6.33","39.43,-6.43","39.43,-6.41",
                       "39.43,-6.39","39.43,-6.37","39.43,-6.35","39.43,-6.33"]

p2_grid_city_lonlat = ["39.53,-6.41","39.53,-6.39","39.53,-6.37","39.53,-6.35","39.53,-6.33","39.53,-6.31","39.51,-6.41","39.51,-6.39",
                       "39.51,-6.37","39.51,-6.35","39.51,-6.33","39.51,-6.31","39.49,-6.41","39.49,-6.39","39.49,-6.37","39.49,-6.35",
                       "39.49,-6.33","39.49,-6.31","39.47,-6.41","39.47,-6.39","39.47,-6.37","39.47,-6.35","39.47,-6.33","39.47,-6.31",
                       "39.45,-6.41","39.45,-6.39","39.45,-6.37","39.45,-6.35","39.45,-6.33","39.45,-6.31"]


token = "0"  # "47602b905233fd"  #"f4041d19f90c7b"  # "ad2bfb8b2f5a7e"
city_cells = []
grids_cells = []

for i in range(0, len(p1_grid_city_lonlat)):
    current_grid_cells, response_json = get_cells_from_opencellid(p1_grid_city_lonlat[i], p2_grid_city_lonlat[i], token, "None")
    grids_cells.append(current_grid_cells)

    with open("/home/jricopal/Escritorio/opencellid_caceres_"+str(i)+".json", 'w') as json_file:
        json.dump(response_json, json_file)

for i in range(0, len(grids_cells)):
    for j in range(0, len(grids_cells[i])):
        city_cells.append(grids_cells[i][j])

print(city_cells)

text_file = open("/home/jricopal/Escritorio/cells_caceres.txt", "w")
text_file.write(str(city_cells))
text_file.close()
    #city_cells_x = []
    #city_cells_y = []

    #for i in range(0, len(city_cells)):
        #city_cells_x.append(city_cells[i][0])
        #city_cells_y.append(city_cells[i][1])

    #plt.axis([0, 2000, 0, 2000])
    #plt.savefig("/home/jricopal/Escritorio/cells_xy_grid_"+str(i)+".png")

{'count': 7, 'cells': [{'lat': 39.515075683594, 'lon': -6.4215087890625, 'mcc': 214, 'mnc': 7, 'lac': 1061, 'cellid': 7220313, 'averageSignalStrength': 0, 'range': 1000, 'samples': 1, 'changeable': 1, 'radio': 'UMTS', 'rnc': 0, 'cid': 0, 'tac': 0, 'sid': 0, 'nid': 0, 'bid': 0}, {'lat': 39.515075683594, 'lon': -6.4215087890625, 'mcc': 214, 'mnc': 7, 'lac': 1061, 'cellid': 13773916, 'averageSignalStrength': 0, 'range': 1000, 'samples': 1, 'changeable': 1, 'radio': 'UMTS', 'rnc': 0, 'cid': 0, 'tac': 0, 'sid': 0, 'nid': 0, 'bid': 0}, {'lat': 39.515075683594, 'lon': -6.4215087890625, 'mcc': 214, 'mnc': 7, 'lac': 1061, 'cellid': 7222538, 'averageSignalStrength': 0, 'range': 1000, 'samples': 1, 'changeable': 1, 'radio': 'UMTS', 'rnc': 0, 'cid': 0, 'tac': 0, 'sid': 0, 'nid': 0, 'bid': 0}, {'lat': 39.515075683594, 'lon': -6.4215087890625, 'mcc': 214, 'mnc': 7, 'lac': 10610, 'cellid': 25605919, 'averageSignalStrength': 0, 'range': 1000, 'samples': 1, 'changeable': 1, 'radio': 'LTE', 'rnc': 0, 'c